# Detector 2D FPN v2 — Kaggle T4 với data.zip

Notebook này nhận cả hai dạng Kaggle Input: dataset lớn còn file data/data.zip, hoặc dataset đã được Kaggle giải nén thành data/data/img_data. Không cần giải nén trước trên máy local.

detector_2d_fpn_v2 là tên kiến trúc bên trong detector_2d.py, không phải tên file. Checkpoint sau train sẽ là best_detector_2d.pth.

Cách dùng khuyến nghị:

1. Add dataset lớn của bạn — bên trong có data/data.zip;
2. Nếu dataset lớn chưa chứa code/labels/weights, Add thêm dataset code/weights;
3. Notebook tự nhận dạng ZIP hoặc thư mục data/data/img_data; nếu là ZIP thì giải nén riêng ảnh vào /kaggle/working và bỏ qua video.

Lưu ý: chỉ có ảnh trong data.zip thì chưa đủ để train; vẫn cần code detector_2d.py, train_week6.py, train_detector_2d.py và dataset_labels (1).json. Nếu đã gộp các file này vào cùng Kaggle Dataset lớn thì chỉ cần Add đúng một dataset đó.

In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU. Vào Settings → Accelerator → GPU T4 rồi Restart Session.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
if importlib.util.find_spec('timm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
import timm
print('timm:', timm.__version__)

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
OUTPUT_DIR = WORK_ROOT / 'detector_2d_v2'
UNPACK_ROOT = WORK_ROOT / 'fire_detection_cctv_data'

# Để None để tự dò. Chỉ điền DATASET_INPUT_HINT nếu bạn muốn khóa vào
# đúng Kaggle Dataset lớn chứa data/data.zip.
DATASET_INPUT_HINT = None
CODE_ROOT_HINT = None
LABELS_HINT = None
DATA_ZIP_HINT = None
INIT_CHECKPOINT_HINT = None
OLD_CHECKPOINT_HINT = None
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def unique_paths(paths):
    result, seen = [], set()
    for value in paths:
        path = Path(value)
        key = str(path).lower()
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result

print('Kaggle input roots:')
for path in sorted(INPUT_ROOT.iterdir()):
    print(' ', path)
print('ZIP files:')
for path in sorted(INPUT_ROOT.rglob('*.zip')):
    print(' ', path, path.stat().st_size, 'bytes')
print('Code candidates:')
for path in sorted(INPUT_ROOT.rglob('detector_2d.py')):
    print(' ', path.parent)
print('Label candidates:')
for path in sorted(INPUT_ROOT.rglob('dataset_labels (1).json')):
    print(' ', path)

In [ ]:
def find_code_root():
    if CODE_ROOT_HINT is not None:
        candidate = Path(CODE_ROOT_HINT).resolve()
        if all((candidate / name).is_file() for name in ('detector_2d.py', 'train_week6.py', 'train_detector_2d.py')):
            return candidate
        raise FileNotFoundError(f'CODE_ROOT_HINT không hợp lệ: {candidate}')
    candidates = [path.parent for path in INPUT_ROOT.rglob('detector_2d.py') if all((path.parent / name).is_file() for name in ('train_week6.py', 'train_detector_2d.py'))]
    candidates = unique_paths(candidates)
    if not candidates:
        raise FileNotFoundError('Không tìm thấy code. Hãy Add Input chứa detector_2d.py, train_week6.py và train_detector_2d.py.')
    candidates.sort(key=lambda path: ('sam' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

def find_labels():
    if LABELS_HINT is not None:
        path = Path(LABELS_HINT).resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates = unique_paths(INPUT_ROOT.rglob('dataset_labels (1).json'))
    if not candidates:
        raise FileNotFoundError('Không tìm thấy dataset_labels (1).json dưới /kaggle/input.')
    candidates.sort(key=lambda path: ('fire-model-data' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

def find_data_input():
    # Trường hợp 1: người dùng muốn chỉ định trực tiếp data.zip.
    if DATA_ZIP_HINT is not None:
        path = Path(DATA_ZIP_HINT).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return ('zip', path)

    search_root = INPUT_ROOT
    if DATASET_INPUT_HINT is not None:
        search_root = Path(DATASET_INPUT_HINT).expanduser().resolve()
        if not search_root.is_dir():
            raise FileNotFoundError(f'DATASET_INPUT_HINT không hợp lệ: {search_root}')

    # Trường hợp 2: Kaggle Input vẫn giữ nguyên file ZIP.
    zip_candidates = unique_paths(search_root.rglob('data.zip'))
    if zip_candidates:
        zip_candidates.sort(key=lambda path: ('data/data.zip' not in str(path).lower(), 'fire-detection-from-cctv' not in str(path).lower(), len(path.parts), str(path).lower()))
        return ('zip', zip_candidates[0])

    # Trường hợp 3: Kaggle Dataset đã giải nén thành data/data/img_data.
    layout_candidates = []
    for candidate in unique_paths([p for p in search_root.rglob('*') if p.is_dir()]):
        for root in (candidate, candidate / 'data'):
            image_root = root / 'img_data'
            if (image_root / 'train').is_dir() and (image_root / 'test').is_dir():
                layout_candidates.append(root.parent if root.name == 'data' else root)
    layout_candidates = unique_paths(layout_candidates)
    if layout_candidates:
        layout_candidates.sort(key=lambda path: ('sam-detector2d-input' not in str(path).lower(), len(path.parts), str(path).lower()))
        return ('directory', layout_candidates[0])

    raise FileNotFoundError(f'Không tìm thấy data.zip hoặc thư mục data/img_data dưới {search_root}. Kiểm tra dataset đã được Add Input chưa.')

def find_optional(name, hint):
    if hint is not None:
        path = Path(hint).resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates = unique_paths(INPUT_ROOT.rglob(name))
    if not candidates:
        return None
    candidates.sort(key=lambda path: ('fire-model-data' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

CODE_ROOT = find_code_root()
CODE_SCRIPT = CODE_ROOT / 'train_detector_2d.py'
LABELS = find_labels()
DATA_INPUT_KIND, DATA_INPUT = find_data_input()
INIT_CHECKPOINT = find_optional('best_spatial.pth', INIT_CHECKPOINT_HINT)
OLD_CHECKPOINT = find_optional('best.pth', OLD_CHECKPOINT_HINT)
print('CODE_ROOT =', CODE_ROOT)
print('CODE_SCRIPT =', CODE_SCRIPT)
print('LABELS =', LABELS)
print('DATASET_INPUT_HINT =', DATASET_INPUT_HINT)
print('DATA_INPUT_KIND =', DATA_INPUT_KIND)
print('DATA_INPUT =', DATA_INPUT)
print('INIT_CHECKPOINT =', INIT_CHECKPOINT)
print('OLD_CHECKPOINT =', OLD_CHECKPOINT)
assert CODE_SCRIPT.is_file(), CODE_SCRIPT
assert LABELS.is_file(), LABELS

## Giải nén data.zip

ZIP của bạn có các member dạng data/img_data/train/... và data/img_data/test/.... Cell dưới đây chỉ giải nén ảnh, bỏ qua data/video_data để tiết kiệm thời gian.

In [ ]:
def image_member(name):
    normalized = name.replace('\\\\', '/').lstrip('/')
    marker = 'data/img_data/'
    position = normalized.lower().find(marker)
    if position < 0:
        return None
    relative = normalized[position:]
    if Path(relative).suffix.lower() not in IMAGE_EXTENSIONS:
        return None
    return relative

def unpack_images(zip_path, output_root):
    train_dir = output_root / 'data' / 'img_data' / 'train'
    test_dir = output_root / 'data' / 'img_data' / 'test'
    if train_dir.is_dir() and test_dir.is_dir():
        print('Đã giải nén từ lần chạy trước:', output_root)
        return output_root
    output_root.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            relative = image_member(info.filename)
            if relative is None:
                continue
            destination = (output_root / relative).resolve()
            try:
                destination.relative_to(output_root.resolve())
            except ValueError as exc:
                raise RuntimeError(f'ZIP member không an toàn: {info.filename}') from exc
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, 'r') as source, destination.open('wb') as target:
                shutil.copyfileobj(source, target)
            count += 1
    print('Đã giải nén ảnh:', count)
    if count == 0 or not train_dir.is_dir() or not test_dir.is_dir():
        raise RuntimeError('data.zip không có đúng data/img_data/train và data/img_data/test')
    return output_root

if DATA_INPUT_KIND == 'zip':
    DATASET_ROOT = unpack_images(DATA_INPUT, UNPACK_ROOT)
else:
    # Dataset đã được Kaggle giải nén; dùng trực tiếp, không copy lại.
    DATASET_ROOT = DATA_INPUT
    print('Dùng thư mục ảnh trực tiếp:', DATASET_ROOT)
for split in ('train', 'test', 'val'):
    split_root = DATASET_ROOT / 'data' / 'img_data' / split
    if split_root.is_dir():
        count = sum(1 for path in split_root.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
        print(split, 'images =', count)

In [ ]:
sys.path.insert(0, str(CODE_ROOT))
import detector_2d
print('ARCHITECTURE =', detector_2d.ARCHITECTURE)
print('DEFAULT_IMAGE_SIZE =', detector_2d.DEFAULT_IMAGE_SIZE)
assert detector_2d.ARCHITECTURE == 'detector_2d_fpn_v2'
assert detector_2d.DEFAULT_IMAGE_SIZE == (640, 640)

from train_week6 import load_records, split_records
records, load_stats = load_records(LABELS, DATASET_ROOT)
splits = split_records(records, seed=42)
split_info = {name: {'count': len(items), 'fire': sum(int(item.has_fire) for item in items), 'no_fire': sum(int(not item.has_fire) for item in items)} for name, items in splits.items()}
print('load_stats =', load_stats)
print('split_info =', split_info)
if not records or any(value['count'] == 0 for value in split_info.values()):
    raise RuntimeError('Không tạo được đủ train/val/test split. Kiểm tra labels và data.zip.')

## Train detector 2D mới

Cấu hình T4: input 640×640, batch 4, gradient accumulation 2, batch hiệu dụng 8, 35 epochs. Nếu hết VRAM, đổi batch thành 2 và accumulation thành 4.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(CODE_SCRIPT),
    '--labels', str(LABELS),
    '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--device', '0',
    '--image-size', '640',
    '--batch-size', '4',
    '--grad-accumulation', '2',
    '--epochs', '35',
    '--freeze-epochs', '3',
    '--workers', '2',
]
if INIT_CHECKPOINT is not None:
    command.extend(['--init-checkpoint', str(INIT_CHECKPOINT)])
else:
    command.append('--no-pretrained')
print('Running:')
print(' '.join(str(item) for item in command))
subprocess.run(command, check=True)

In [ ]:
metrics_path = OUTPUT_DIR / 'test_metrics.json'
history_path = OUTPUT_DIR / 'history.json'
best_path = OUTPUT_DIR / 'best_detector_2d.pth'
for path in (metrics_path, history_path, best_path):
    print(path, path.exists(), path.stat().st_size if path.exists() else None)
if not metrics_path.is_file() or not best_path.is_file():
    raise FileNotFoundError('Train chưa tạo đủ best_detector_2d.pth và test_metrics.json')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt
history = json.loads(history_path.read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
train_loss = [row['train']['total'] for row in history]
val_loss = [row['val']['total'] for row in history]
val_f1 = [row['val_metric']['f1'] for row in history]
val_mae = [row['val_metric']['mae_px'] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs, train_loss, label='train')
axes[0].plot(epochs, val_loss, label='val')
axes[0].set_title('Total loss')
axes[0].legend()
axes[0].grid()
axes[1].plot(epochs, val_f1)
axes[1].set_title('Validation F1')
axes[1].grid()
axes[2].plot(epochs, val_mae)
axes[2].set_title('Validation MAE pixel')
axes[2].grid()
plt.show()

## So sánh với detector cũ (tùy chọn)

Cell này chỉ chạy nếu input có best.pth và compare_detector_2d.py.

In [ ]:
COMPARE_SCRIPT = CODE_ROOT / 'compare_detector_2d.py'
COMPARE_OUTPUT = WORK_ROOT / 'detector_2d_comparison'
if OLD_CHECKPOINT is None or not OLD_CHECKPOINT.is_file():
    print('Bỏ qua comparison: chưa upload best.pth.')
elif not COMPARE_SCRIPT.is_file():
    print('Bỏ qua comparison: chưa upload compare_detector_2d.py.')
else:
    compare_command = [
        sys.executable, str(COMPARE_SCRIPT),
        '--labels', str(LABELS),
        '--dataset', str(DATASET_ROOT),
        '--old', str(OLD_CHECKPOINT),
        '--new', str(best_path),
        '--split', 'test',
        '--max-images', '0',
        '--threshold', '0.5',
        '--device', 'cuda:0',
        '--output-dir', str(COMPARE_OUTPUT),
    ]
    print(' '.join(str(item) for item in compare_command))
    subprocess.run(compare_command, check=True)
    summary_path = COMPARE_OUTPUT / 'summary.json'
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        for branch, values in summary.get('branches', {}).items():
            print(branch, {key: values.get(key) for key in ('precision', 'recall', 'f1', 'mae_px', 'median_px', 'p95_px', 'pck10', 'pck25', 'latency_ms_mean')})

In [ ]:
from IPython.display import display, FileLink, Image as IPImage
preview = COMPARE_OUTPUT / 'detector_2d_contact_sheet.png'
if preview.is_file():
    display(IPImage(filename=str(preview)))
else:
    print('Chưa có contact sheet comparison.')
archive_path = Path(shutil.make_archive(str(WORK_ROOT / 'detector_2d_v2', 'zip'), 'zip', str(OUTPUT_DIR)))
print('Download:', archive_path)
display(FileLink(str(archive_path)))